# Проведение экспериментов по настройке модели

Задача — бинарная классификация `target` по клиническим показателям пациента.
Исходные данные — очищенная выборка из ЛР1 (`../data/clean_data.pkl`, 296 записей).

Все эксперименты логируются в MLflow. Перед запуском блокнота поднимите сервер:

```shell
cd ../mlflow && sh start_mlflow.sh
```

## Загрузка модулей

In [1]:
import pickle
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             roc_auc_score)
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (KBinsDiscretizer, PolynomialFeatures,
                                   StandardScaler, TargetEncoder)

warnings.filterwarnings('ignore')
RANDOM_STATE = 42

## Загрузка данных

In [2]:
with open('../data/clean_data.pkl', 'rb') as f:
    data = pickle.load(f)

print('Размер выборки:', data.shape)
data.info()

Размер выборки: (296, 15)
<class 'pandas.DataFrame'>
Index: 296 entries, 0 to 302
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   age        296 non-null    int8    
 1   sex        296 non-null    int8    
 2   cp         296 non-null    int8    
 3   trestbps   296 non-null    int16   
 4   chol       296 non-null    int16   
 5   fbs        296 non-null    int8    
 6   restecg    296 non-null    int8    
 7   thalach    296 non-null    int16   
 8   exang      296 non-null    int8    
 9   oldpeak    296 non-null    float32 
 10  slope      296 non-null    int8    
 11  ca         296 non-null    int8    
 12  thal       296 non-null    int8    
 13  target     296 non-null    int8    
 14  age_group  296 non-null    category
dtypes: category(1), float32(1), int16(3), int8(10)
memory usage: 8.4 KB


Признак `age_group` был создан в ЛР1 для визуализации и является производным от `age`.
В модель он не подаётся: это тот же возраст, только огрублённый, и он вносил бы
избыточность. Удаляем его из рабочей выборки.

In [3]:
data = data.drop(columns=['age_group'])

numeric_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
categorical_features = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
target = 'target'

print('Числовых признаков     :', len(numeric_features), numeric_features)
print('Категориальных признаков:', len(categorical_features), categorical_features)

Числовых признаков     : 5 ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
Категориальных признаков: 8 ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']


## Разбиение на обучающую и тестовую выборки

In [4]:
X = data.drop(columns=target)
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.75, random_state=RANDOM_STATE, stratify=y
)

print(f'Обучающая: {X_train.shape[0]} записей')
print(f'Тестовая : {X_test.shape[0]} записей')
print()
print('Баланс классов в обучающей:')
print(y_train.value_counts(normalize=True).round(3).to_string())
print('Баланс классов в тестовой:')
print(y_test.value_counts(normalize=True).round(3).to_string())

Обучающая: 222 записей
Тестовая : 74 записей

Баланс классов в обучающей:
target
1    0.541
0    0.459
Баланс классов в тестовой:
target
1    0.541
0    0.459


Разбиение 75/25, как требует задание. Дополнительно использована **стратификация**
по целевой переменной (`stratify=y`) — она сохраняет соотношение классов в обеих частях.
На выборке в 296 записей без неё баланс мог заметно съехать случайным образом,
и метрики на тесте стали бы плохо сопоставимы между экспериментами.

## Baseline-модель

Пайплайн из двух шагов, как требует задание:

- `transform` — `ColumnTransformer`: `StandardScaler` для числовых признаков,
  `TargetEncoder` для категориальных (задача классификации);
- `classification` — `RandomForestClassifier` с параметрами по умолчанию.

Случайный лес в шкалировании не нуждается, но шаг оставлен намеренно: пайплайн должен
оставаться пригодным и для других алгоритмов.

In [5]:
preprocessor = ColumnTransformer(transformers=[
    ('numeric', StandardScaler(), numeric_features),
    ('categorical', TargetEncoder(), categorical_features),
])

pipeline_baseline = Pipeline([
    ('transform', preprocessor),
    ('classification', RandomForestClassifier(random_state=RANDOM_STATE)),
])

pipeline_baseline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('transform', ...), ('classification', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float,

In [6]:
def calc_metrics(model, X, y_true):
    """Метрики качества классификации на тестовой выборке."""
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)[:, 1]
    return {
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
        'roc_auc': roc_auc_score(y_true, y_proba),
    }


def print_metrics(metrics, title=''):
    if title:
        print(title)
    for name, value in metrics.items():
        print(f'  {name:<10} {value:.4f}')

In [7]:
pipeline_baseline.fit(X_train, y_train)
metrics_baseline = calc_metrics(pipeline_baseline, X_test, y_test)
print_metrics(metrics_baseline, 'Baseline (RandomForest, параметры по умолчанию):')

Baseline (RandomForest, параметры по умолчанию):
  precision  0.7619
  recall     0.8000
  f1         0.7805
  roc_auc    0.8518


`roc_auc` считается по вероятностям (`predict_proba`), а не по меткам — иначе метрика
вырождается и теряет смысл. Остальные три метрики считаются по предсказанным меткам
с порогом 0.5.

## Подключение MLflow

In [8]:
import mlflow
from mlflow.models import infer_signature

TRACKING_URI = 'http://127.0.0.1:5000'
EXPERIMENT_NAME = 'Heart Disease Classification'

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_registry_uri(TRACKING_URI)

REGISTERED_MODEL_NAME = 'heart-disease-rf'
INPUT_EXAMPLE = X_train.head(5)
SIGNATURE = infer_signature(model_input=X_train.head(5))

print('MLflow tracking URI:', mlflow.get_tracking_uri())
print('Версия MLflow      :', mlflow.__version__)

MLflow tracking URI: http://127.0.0.1:5000
Версия MLflow      : 3.16.1


### Функция логирования

Вынесена отдельно, чтобы каждый эксперимент логировался единообразно. Вместе с моделью
логируются сигнатура, пример входных данных и файл зависимостей — как требует п.9 задания.

In [9]:
def log_experiment(run_name, model, metrics=None, params=None,
                   artifacts=None, register=False, signature=SIGNATURE):
    """Логирует модель и сопутствующие артефакты в MLflow."""
    experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    experiment_id = (experiment.experiment_id if experiment
                     else mlflow.create_experiment(EXPERIMENT_NAME))

    with mlflow.start_run(run_name=run_name, experiment_id=experiment_id) as run:
        if metrics:
            mlflow.log_metrics(metrics)
        if params:
            mlflow.log_params(params)
        for path in (artifacts or []):
            mlflow.log_artifact(path)

        mlflow.sklearn.log_model(
            sk_model=model,
            name='model',
            signature=signature,
            input_example=INPUT_EXAMPLE,
            pip_requirements='../requirements.txt',
            registered_model_name=REGISTERED_MODEL_NAME if register else None,
            # MLflow 3 по умолчанию сериализует через skops, который отказывается
            # сохранять деревья решений и пользовательские классы без явного
            # перечисления доверенных типов. Пайплайн содержит и то, и другое,
            # поэтому используем cloudpickle — он работает с произвольными
            # объектами и остаётся штатным форматом MLflow.
            serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
        )
        print(f'{run_name}: run_id = {run.info.run_id}')
        return run.info.run_id

In [10]:
run_id_baseline = log_experiment(
    'baseline_random_forest',
    pipeline_baseline,
    metrics=metrics_baseline,
    params={'model': 'RandomForestClassifier', 'features': 'исходные 13'},
    register=True,
)

2026/09/28 18:40:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Successfully registered model 'heart-disease-rf'.
2026/09/28 18:40:32 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: heart-disease-rf, version 1


baseline_random_forest: run_id = 7cd0d359d13442a4ac53461717625314
🏃 View run baseline_random_forest at: http://127.0.0.1:5000/#/experiments/1/runs/7cd0d359d13442a4ac53461717625314
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '1' of model 'heart-disease-rf'.


Модель зарегистрирована в реестре как **версия 1** (`registered_model_name`).
Проверить можно в интерфейсе: вкладка *Models* → `heart-disease-rf`.

## Генерация новых признаков средствами sklearn

Применяем два преобразования:

- **`PolynomialFeatures(degree=2)`** к признакам `age`, `thalach`, `oldpeak`. В ЛР1
  было показано, что разделяющая сила `thalach` зависит от возраста — линии тренда
  классов сходятся к 70 годам. Полиномиальные признаки как раз создают произведение
  `age × thalach`, то есть дают модели это взаимодействие явно.
- **`KBinsDiscretizer`** к `trestbps` и `chol`. Оба признака показали почти нулевую
  корреляцию с целевой, но связь может быть нелинейной — например, опасны и слишком
  низкие, и слишком высокие значения. Разбиение на корзины позволяет модели уловить
  такую зависимость.

Полученные полиномиальные признаки сами требуют шкалирования — поэтому каждое
преобразование обёрнуто в мини-пайплайн с `StandardScaler` на конце.

In [11]:
poly_features = ['age', 'thalach', 'oldpeak']
kbins_features = ['trestbps', 'chol']

poly_transformer = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scale', StandardScaler()),
])

kbins_transformer = Pipeline([
    ('kbins', KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile')),
    ('scale', StandardScaler()),
])

extended_transformer = ColumnTransformer(transformers=[
    ('numeric', StandardScaler(), numeric_features),
    ('categorical', TargetEncoder(), categorical_features),
    ('poly', poly_transformer, poly_features),
    ('kbins', kbins_transformer, kbins_features),
])

In [12]:
X_train_fe_sklearn = X_train.copy()
X_train_fe_sklearn = extended_transformer.fit_transform(X_train_fe_sklearn, y_train)

feature_names = list(extended_transformer.get_feature_names_out())

print(f'Признаков было : {X_train.shape[1]}')
print(f'Признаков стало: {X_train_fe_sklearn.shape[1]}')
print()
print('Новые признаки:')
for name in feature_names[len(numeric_features) + len(categorical_features):]:
    print(' ', name)

Признаков было : 13
Признаков стало: 24

Новые признаки:
  poly__age
  poly__thalach
  poly__oldpeak
  poly__age^2
  poly__age thalach
  poly__age oldpeak
  poly__thalach^2
  poly__thalach oldpeak
  poly__oldpeak^2
  kbins__trestbps
  kbins__chol


In [13]:
FEATURES_PATH = 'features_after_engineering.txt'
with open(FEATURES_PATH, 'w') as f:
    print(*feature_names, sep='\n', file=f)

print(f'Список из {len(feature_names)} столбцов сохранён в {FEATURES_PATH}')

Список из 24 столбцов сохранён в features_after_engineering.txt


In [14]:
pipeline_fe = Pipeline([
    ('transform', extended_transformer),
    ('classification', RandomForestClassifier(random_state=RANDOM_STATE)),
])

pipeline_fe.fit(X_train, y_train)
metrics_fe = calc_metrics(pipeline_fe, X_test, y_test)
print_metrics(metrics_fe, 'После генерации признаков:')
print()
print_metrics(metrics_baseline, 'Baseline для сравнения:')

После генерации признаков:
  precision  0.7111
  recall     0.8000
  f1         0.7529
  roc_auc    0.8349

Baseline для сравнения:
  precision  0.7619
  recall     0.8000
  f1         0.7805
  roc_auc    0.8518


In [15]:
run_id_fe = log_experiment(
    'feature_engineering_sklearn',
    pipeline_fe,
    metrics=metrics_fe,
    params={'model': 'RandomForestClassifier',
            'poly_features': ', '.join(poly_features),
            'kbins_features': ', '.join(kbins_features),
            'features_total': len(feature_names)},
    artifacts=[FEATURES_PATH],
)

2026/09/28 18:40:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


feature_engineering_sklearn: run_id = 534108936a824a3989ed680a914e08ca
🏃 View run feature_engineering_sklearn at: http://127.0.0.1:5000/#/experiments/1/runs/534108936a824a3989ed680a914e08ca
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


## Отбор наиболее значимых признаков

Отбор проводим на расширенном наборе из предыдущего пункта. Из полученных признаков
оставляем **половину** — это середина рекомендованного заданием диапазона 20–70%.

Используется `SequentialFeatureSelector` из `mlxtend` с последовательным **добавлением**
признаков (`forward=True`): алгоритм начинает с пустого набора и на каждом шаге добавляет
тот признак, который сильнее всего улучшает метрику на кросс-валидации.

In [16]:
from mlxtend.feature_selection import SequentialFeatureSelector as SFS

n_features_to_select = X_train_fe_sklearn.shape[1] // 2
print(f'Отбираем {n_features_to_select} признаков из {X_train_fe_sklearn.shape[1]}')

sfs = SFS(
    RandomForestClassifier(n_estimators=50, random_state=RANDOM_STATE),
    k_features=n_features_to_select,
    forward=True,
    floating=False,
    scoring='f1',
    cv=3,
    n_jobs=-1,
)
sfs = sfs.fit(X_train_fe_sklearn, y_train)

selected_idx = list(sfs.k_feature_idx_)
selected_names = [feature_names[i] for i in selected_idx]

print(f'\nЛучшая метрика f1 на кросс-валидации: {sfs.k_score_:.4f}')
print('\nОтобранные признаки:')
for i, name in zip(selected_idx, selected_names):
    print(f'  [{i:2}] {name}')

Отбираем 12 признаков из 24



Лучшая метрика f1 на кросс-валидации: 0.8628

Отобранные признаки:
  [ 1] numeric__trestbps
  [ 2] numeric__chol
  [ 4] numeric__oldpeak
  [ 6] categorical__cp
  [ 9] categorical__exang
  [10] categorical__slope
  [11] categorical__ca
  [12] categorical__thal
  [18] poly__age oldpeak
  [20] poly__thalach oldpeak
  [21] poly__oldpeak^2
  [23] kbins__chol


In [17]:
SELECTED_NAMES_PATH = 'selected_features_names.txt'
SELECTED_IDX_PATH = 'selected_features_indices.txt'

with open(SELECTED_NAMES_PATH, 'w') as f:
    print(*selected_names, sep='\n', file=f)
with open(SELECTED_IDX_PATH, 'w') as f:
    print(*selected_idx, sep=',', file=f)

print('Сохранено:', SELECTED_NAMES_PATH, 'и', SELECTED_IDX_PATH)

Сохранено: selected_features_names.txt и selected_features_indices.txt


Чтобы встроить отбор в пайплайн вторым шагом, нужен трансформер, который оставляет
столбцы по индексам. Пишем минимальный совместимый с sklearn класс.

In [18]:
from sklearn.base import BaseEstimator, TransformerMixin


class IndexSelector(BaseEstimator, TransformerMixin):
    """Оставляет столбцы с заданными индексами.

    Конструктор сохраняет параметр ровно в том виде, в каком его передали:
    sklearn при клонировании (а это делает cross_val_score) проверяет, что
    __init__ не меняет свои аргументы, иначе клонирование падает.
    """

    def __init__(self, indices):
        self.indices = indices

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return np.asarray(X)[:, list(self.indices)]

In [19]:
pipeline_fs = Pipeline([
    ('transform', extended_transformer),
    ('feature_selection', IndexSelector(selected_idx)),
    ('classification', RandomForestClassifier(random_state=RANDOM_STATE)),
])

pipeline_fs.fit(X_train, y_train)
metrics_fs = calc_metrics(pipeline_fs, X_test, y_test)
print_metrics(metrics_fs, f'После отбора {n_features_to_select} признаков:')

После отбора 12 признаков:
  precision  0.7500
  recall     0.8250
  f1         0.7857
  roc_auc    0.8364


In [20]:
run_id_fs = log_experiment(
    'feature_selection_mlxtend',
    pipeline_fs,
    metrics=metrics_fs,
    params={'model': 'RandomForestClassifier',
            'selector': 'SFS forward (mlxtend)',
            'features_selected': len(selected_idx),
            'features_total': len(feature_names)},
    artifacts=[SELECTED_NAMES_PATH, SELECTED_IDX_PATH],
)

2026/09/28 18:40:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


feature_selection_mlxtend: run_id = 6005cdc8cfb34f5bb5fad013c71736d7
🏃 View run feature_selection_mlxtend at: http://127.0.0.1:5000/#/experiments/1/runs/6005cdc8cfb34f5bb5fad013c71736d7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


### Сравнение с другим алгоритмом отбора *(п. 13, со звёздочкой)*

Повторим отбор методом `RFE` — он идёт с противоположной стороны: начинает с полного
набора и последовательно **удаляет** наименее важные признаки. Интересно, совпадут ли
выборы двух алгоритмов.

In [21]:
rfe = RFE(
    RandomForestClassifier(n_estimators=50, random_state=RANDOM_STATE),
    n_features_to_select=n_features_to_select,
)
rfe.fit(X_train_fe_sklearn, y_train)

rfe_idx = list(np.where(rfe.support_)[0])
rfe_names = [feature_names[i] for i in rfe_idx]

common = sorted(set(selected_idx) & set(rfe_idx))
only_sfs = sorted(set(selected_idx) - set(rfe_idx))
only_rfe = sorted(set(rfe_idx) - set(selected_idx))

print(f'SFS  отобрал: {len(selected_idx)} признаков')
print(f'RFE  отобрал: {len(rfe_idx)} признаков')
print(f'Совпало     : {len(common)} ({len(common) / n_features_to_select:.0%})')
print()
print('Выбрали оба:')
for i in common:
    print(f'  [{i:2}] {feature_names[i]}')
print('\nТолько SFS:')
for i in only_sfs:
    print(f'  [{i:2}] {feature_names[i]}')
print('\nТолько RFE:')
for i in only_rfe:
    print(f'  [{i:2}] {feature_names[i]}')

SFS  отобрал: 12 признаков
RFE  отобрал: 12 признаков
Совпало     : 8 (67%)

Выбрали оба:
  [ 2] numeric__chol
  [ 6] categorical__cp
  [10] categorical__slope
  [11] categorical__ca
  [12] categorical__thal
  [18] poly__age oldpeak
  [20] poly__thalach oldpeak
  [21] poly__oldpeak^2

Только SFS:
  [ 1] numeric__trestbps
  [ 4] numeric__oldpeak
  [ 9] categorical__exang
  [23] kbins__chol

Только RFE:
  [ 3] numeric__thalach
  [14] poly__thalach
  [17] poly__age thalach
  [19] poly__thalach^2


## Подбор гиперпараметров с помощью optuna

Задание требует настраивать **ту модель, которая показала лучший результат**. На этот
момент лучшая по f1 — конфигурация с отбором признаков, поэтому именно её структуру
(расширенный набор → отбор 12 признаков → случайный лес) и оптимизируем.

Настраиваем три параметра случайного леса: количество деревьев, глубину и долю
признаков `max_features` в диапазоне 0.1–1.0.

Оптимизируем **f1** (для классификации) и явно указываем `direction='maximize'` —
метрику нужно максимизировать. Качество оцениваем кросс-валидацией на обучающей
выборке, а не на тесте: иначе подбор параметров подгонится под тест, и итоговая
оценка качества перестанет быть честной.

In [22]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)


def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'max_features': trial.suggest_float('max_features', 0.1, 1.0),
    }
    pipeline = Pipeline([
        ('transform', extended_transformer),
        ('feature_selection', IndexSelector(selected_idx)),
        ('classification', RandomForestClassifier(random_state=RANDOM_STATE, **params)),
    ])
    scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='f1', n_jobs=-1)
    return scores.mean()


study = optuna.create_study(direction='maximize', study_name='rf_hyperparams')
study.optimize(objective, n_trials=15, show_progress_bar=False)

print(f'Проведено испытаний: {len(study.trials)}')
print(f'Лучшее значение f1 на кросс-валидации: {study.best_value:.4f}')
print('\nЛучшие параметры:')
for name, value in study.best_params.items():
    print(f'  {name} = {value}')

Проведено испытаний: 15
Лучшее значение f1 на кросс-валидации: 0.8535

Лучшие параметры:
  n_estimators = 114
  max_depth = 10
  max_features = 0.7411742386209574


In [23]:
history = pd.DataFrame([
    {'trial': t.number, 'f1_cv': t.value, **t.params} for t in study.trials
]).sort_values('f1_cv', ascending=False)

history.head(10).round(4)

,trial,f1_cv,n_estimators,max_depth,max_features
9,9,0.8535,114,10,0.7412
4,4,0.8531,175,8,0.1420
8,8,0.8511,156,13,0.3674
6,6,0.8495,64,4,0.5126
11,11,0.8448,149,8,0.1813
3,3,0.8433,138,14,0.6737
12,12,0.8426,214,7,0.4741
13,13,0.8400,63,11,0.2201
14,14,0.8395,171,5,0.8123
5,5,0.8371,206,14,0.6608


In [24]:
pipeline_tuned = Pipeline([
    ('transform', extended_transformer),
    ('feature_selection', IndexSelector(selected_idx)),
    ('classification', RandomForestClassifier(random_state=RANDOM_STATE, **study.best_params)),
])

pipeline_tuned.fit(X_train, y_train)
metrics_tuned = calc_metrics(pipeline_tuned, X_test, y_test)
print_metrics(metrics_tuned, 'После подбора гиперпараметров:')

После подбора гиперпараметров:
  precision  0.7727
  recall     0.8500
  f1         0.8095
  roc_auc    0.8412


In [25]:
run_id_tuned = log_experiment(
    'optuna_tuned_random_forest',
    pipeline_tuned,
    metrics=metrics_tuned,
    params={'model': 'RandomForestClassifier',
            'tuning': 'optuna, 15 trials, f1 maximize',
            'cv_f1': round(study.best_value, 4),
            **study.best_params},
    artifacts=[FEATURES_PATH, SELECTED_NAMES_PATH],
    register=True,
)

2026/09/28 18:40:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Registered model 'heart-disease-rf' already exists. Creating a new version of this model...
2026/09/28 18:40:41 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: heart-disease-rf, version 2


optuna_tuned_random_forest: run_id = 8a9b43dc758144d989357d3396b932c9
🏃 View run optuna_tuned_random_forest at: http://127.0.0.1:5000/#/experiments/1/runs/8a9b43dc758144d989357d3396b932c9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '2' of model 'heart-disease-rf'.


Модель зарегистрирована как **версия 2**.

## Сравнение всех экспериментов

In [26]:
comparison = pd.DataFrame({
    'baseline': metrics_baseline,
    'feature_engineering': metrics_fe,
    'feature_selection': metrics_fs,
    'optuna_tuned': metrics_tuned,
}).T.round(4)

comparison

,precision,recall,f1,roc_auc
baseline,0.7619,0.800,0.7805,0.8518
feature_engineering,0.7111,0.800,0.7529,0.8349
feature_selection,0.7500,0.825,0.7857,0.8364
optuna_tuned,0.7727,0.850,0.8095,0.8412


In [27]:
best_run = comparison['f1'].idxmax()
print(f'Лучший результат по f1: {best_run} (f1 = {comparison.loc[best_run, "f1"]:.4f})')
print()
print('Метрики лучшей модели:')
print(comparison.loc[best_run].to_string())

Лучший результат по f1: optuna_tuned (f1 = 0.8095)

Метрики лучшей модели:
precision    0.7727
recall       0.8500
f1           0.8095
roc_auc      0.8412


### Вывод

Основная метрика для выбора — **f1**: она балансирует точность и полноту, а классы
в выборке близки к сбалансированным. `roc_auc` смотрим как дополнительную оценку
качества ранжирования, не зависящую от порога.

## Обучение финальной модели на всей выборке

Выбранную конфигурацию обучаем заново — уже на **всей** выборке, а не только на
обучающей части. Метрики здесь не считаем: тестовой выборки больше нет, всё ушло
в обучение. Это нормально для финальной модели, которая пойдёт в продакшен —
качество мы уже оценили на предыдущем шаге.

In [28]:
X_full = pd.concat([X_train, X_test], axis=0)
y_full = pd.concat([y_train, y_test], axis=0)

print(f'Полная выборка: {X_full.shape[0]} записей, {X_full.shape[1]} признаков')

BEST_PIPELINES = {
    'baseline': pipeline_baseline,
    'feature_engineering': pipeline_fe,
    'feature_selection': pipeline_fs,
    'optuna_tuned': pipeline_tuned,
}
production_model = BEST_PIPELINES[best_run]
production_model.fit(X_full, y_full)

print(f'Финальная модель обучена на конфигурации: {best_run}')

Полная выборка: 296 записей, 13 признаков


Финальная модель обучена на конфигурации: optuna_tuned


In [29]:
USED_COLUMNS_PATH = 'production_columns.txt'
with open(USED_COLUMNS_PATH, 'w') as f:
    print(*X_full.columns, sep='\n', file=f)

production_signature = infer_signature(
    model_input=X_full,
    model_output=production_model.predict(X_full.head(5)),
)

production_params = {
    'model_type': 'RandomForestClassifier',
    'training_data': 'full_dataset',
    'best_configuration': best_run,
    'total_samples': len(X_full),
    'features_count': X_full.shape[1],
}
if best_run == 'optuna_tuned':
    production_params |= study.best_params

run_id_production = log_experiment(
    'production_model',
    production_model,
    params=production_params,
    artifacts=[USED_COLUMNS_PATH, FEATURES_PATH],
    signature=production_signature,
    register=True,
)

print()
print('RUN_ID PRODUCTION-МОДЕЛИ:', run_id_production)

2026/09/28 18:40:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Registered model 'heart-disease-rf' already exists. Creating a new version of this model...
2026/09/28 18:40:41 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: heart-disease-rf, version 3


production_model: run_id = 857216a4e3f345b8bb9b53c7c8d61a57
🏃 View run production_model at: http://127.0.0.1:5000/#/experiments/1/runs/857216a4e3f345b8bb9b53c7c8d61a57
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1

RUN_ID PRODUCTION-МОДЕЛИ: 857216a4e3f345b8bb9b53c7c8d61a57


Created version '3' of model 'heart-disease-rf'.


### Установка тега Production в реестре моделей

In [30]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

versions = client.search_model_versions(f"name='{REGISTERED_MODEL_NAME}'")
production_version = max(
    (v for v in versions if v.run_id == run_id_production),
    key=lambda v: int(v.version),
)

client.set_model_version_tag(
    name=REGISTERED_MODEL_NAME,
    version=production_version.version,
    key='Production',
    value='true',
)
client.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME,
    alias='Production',
    version=production_version.version,
)

print(f'Модель {REGISTERED_MODEL_NAME} версии {production_version.version} '
      f'помечена тегом Production')
print(f'run_id: {run_id_production}')
print()
print('Все зарегистрированные версии:')
for v in sorted(versions, key=lambda v: int(v.version)):
    print(f'  версия {v.version}: run_id={v.run_id[:8]}...')

Модель heart-disease-rf версии 3 помечена тегом Production
run_id: 857216a4e3f345b8bb9b53c7c8d61a57

Все зарегистрированные версии:
  версия 1: run_id=7cd0d359...
  версия 2: run_id=8a9b43dc...
  версия 3: run_id=857216a4...


В MLflow 3 для пометки модели используются **теги** и **алиасы** вместо устаревших
стадий (`stage`), которые были в версии 2 и объявлены deprecated. Здесь проставлено
и то, и другое: тег `Production=true` виден в списке версий, алиас `Production`
позволяет обращаться к модели по имени `models:/heart-disease-rf@Production` —
именно так она будет загружаться в сервисе в ЛР3.

## Итоги

In [31]:
print('Сводка по экспериментам:')
print(comparison.to_string())
print()
print(f'Лучшая конфигурация : {best_run}')
print(f'Production run_id   : {run_id_production}')
print(f'Версия в реестре    : {production_version.version}')

Сводка по экспериментам:
                     precision  recall      f1  roc_auc
baseline                0.7619   0.800  0.7805   0.8518
feature_engineering     0.7111   0.800  0.7529   0.8349
feature_selection       0.7500   0.825  0.7857   0.8364
optuna_tuned            0.7727   0.850  0.8095   0.8412

Лучшая конфигурация : optuna_tuned
Production run_id   : 857216a4e3f345b8bb9b53c7c8d61a57
Версия в реестре    : 3


Результаты, которые нужно перенести в README:

- лучшая конфигурация и её метрики — из таблицы сравнения выше;
- `run_id` Production-модели — из вывода последней ячейки;
- параметры модели — из результатов optuna;
- список используемых столбцов — файл `production_columns.txt`.